In [46]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

In [47]:
world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

/tmp/ipykernel_2910355/1609955626.py:1: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [3]:
min_lon = -180
max_lon = 180
min_lat = -30
max_lat = 30
tropics = Polygon([(min_lon, min_lat), (min_lon, max_lat), (max_lon, max_lat, ), (max_lon, min_lat )])


In [48]:
world

,pop_est,continent,name,iso_a3,gdp_md_est,geometry
0,889953.0,Oceania,Fiji,FJI,5496,"MULTIPOLYGON (((180.00000 -16.06713, 180.00000..."
1,58005463.0,Africa,Tanzania,TZA,63177,"POLYGON ((33.90371 -0.95000, 34.07262 -1.05982..."
2,603253.0,Africa,W. Sahara,ESH,907,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,37589262.0,North America,Canada,CAN,1736425,"MULTIPOLYGON (((-122.84000 49.00000, -122.9742..."
4,328239523.0,North America,United States of America,USA,21433226,"MULTIPOLYGON (((-122.84000 49.00000, -120.0000..."
...,...,...,...,...,...,...
172,6944975.0,Europe,Serbia,SRB,51475,"POLYGON ((18.82982 45.90887, 18.82984 45.90888..."
173,622137.0,Europe,Montenegro,MNE,5542,"POLYGON ((20.07070 42.58863, 19.80161 42.50009..."
174,1794248.0,Europe,Kosovo,-99,7926,"POLYGON ((20.59025 41.85541, 20.52295 42.21787..."
175,1394973.0,North America,Trinidad and Tobago,TTO,24269,"POLYGON ((-61.68000 10.76000, -61.10500 10.890..."


In [4]:
tropical_countries = world.loc[world.intersects(tropics)]

In [ ]:
labels = 

In [50]:
df = pd.read_csv('../data/tmp/FAOSTAT_data_en_3-30-2024.csv')

In [52]:
df.Item.unique()

array(['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms'], dtype=object)

In [55]:
from IPython.display import clear_output



for idx, row in tropical_countries.iterrows():
    print(row['name'])
    _df = df.loc[df.Area == row['name']]
    print(_df.shape)
    print(_df.Item.unique())
    _waitforinput = input()

    clear_output(wait=True)    

Fiji
(14, 15)
['Cattle' 'Cocoa beans' 'Coffee, green']


KeyboardInterrupt: Interrupted by user

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
738,QCL,Crops and livestock products,242,Fiji,5111,Stocks,2111.0,Cattle,2010,2010,An,309000.00,E,Estimated value,NaN
739,QCL,Crops and livestock products,242,Fiji,5111,Stocks,2111.0,Cattle,2020,2020,An,93301.00,A,Official figure,NaN
740,QCL,Crops and livestock products,242,Fiji,5312,Area harvested,1640.0,Cocoa beans,2010,2010,ha,48.00,A,Official figure,NaN
741,QCL,Crops and livestock products,242,Fiji,5419,Yield,1640.0,Cocoa beans,2010,2010,100 g/ha,836.00,A,Official figure,NaN
742,QCL,Crops and livestock products,242,Fiji,5510,Production,1640.0,Cocoa beans,2010,2010,t,4.00,A,Official figure,NaN
743,QCL,Crops and livestock products,242,Fiji,5312,Area harvested,1640.0,Cocoa beans,2020,2020,ha,5.00,A,Official figure,NaN
744,QCL,Crops and livestock products,242,Fiji,5419,Yield,1640.0,Cocoa beans,2020,2020,100 g/ha,20000.00,A,Official figure,NaN
745,QCL,Crops and livestock products,242,Fiji,5510,Production,1640.0,Cocoa beans,2020,2020,t,9.10,A,Official figure,NaN
746,QCL,Crops and livestock products,242,Fiji,5312,Area harvested,1610.0,"Coffee, green",2010,2010,ha,29.00,I,Imputed value,NaN
747,QCL,Crops and livestock products,242,Fiji,5419,Yield,1610.0,"Coffee, green",2010,2010,100 g/ha,5261.00,E,Estimated value,NaN


In [34]:
cocoa_fiji = _df.loc[(_df.Element == 'Area harvested') & ( _df.Area == 'Fiji') & ( _df.Item == 'Cocoa beans')]

In [ ]:
#Here the number is negative - 
#ie. area harvested is smaller in 2020 than in 2010; 
#therefore no expansion and i should enter 0 in the model

In [42]:
harvested_2010 = cocoa_fiji.loc[cocoa_fiji.Year == 2010].Value.values[0] 
harvested_2020 = cocoa_fiji.loc[cocoa_fiji.Year == 2020].Value.values[0] 


5.0

In [59]:
FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']

In [62]:


elements = {
    'Cattle' : {}, 
    'Palm oil' : {}, 
    'Soya beans' : {}, 
    'Cocoa beans' : {}, 
    'Coffee, green' : {}, 
    'Natural rubber in primary forms' : {}
}

for idx, row in tropical_countries.iterrows():
    _df = df.loc[df.Area == row['name']]
    for _cls in FAOSTAT_classes:
        commodity_df = _df.loc[_df.Item == _cls]
        elements_available = commodity_df.Element.unique()
        
        for element in elements_available:
            try:
                elements[_cls][element] += 1
            except KeyError:
                elements[_cls][element] = 1

In [63]:
import pprint

In [64]:
pprint.pprint(elements)

{'Cattle': {'Stocks': 96},
 'Cocoa beans': {'Area harvested': 42, 'Production': 42, 'Yield': 41},
 'Coffee, green': {'Area harvested': 60, 'Production': 60, 'Yield': 59},
 'Natural rubber in primary forms': {'Area harvested': 28,
                                     'Production': 28,
                                     'Yield': 24},
 'Palm oil': {'Production': 37},
 'Soya beans': {'Area harvested': 61, 'Production': 61, 'Yield': 55}}


In [76]:
df_filtered = df.loc[
    (df.Item == 'Cattle') |
    ((df.Item == 'Palm oil') & (df.Element == 'Production' )) |
    ((df.Item == 'Cocoa beans') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Coffee, green') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Natural rubber in primary forms') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Soya beans') & (df.Element == 'Area harvested' )) 
]

In [77]:
df_filtered.to_csv('../data/tmp/faostat_reports.csv', index=False)

In [78]:
df_filtered

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2010,2010,An,5673000.0,A,Official figure,NaN
1,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2020,2020,An,5326747.0,E,Estimated value,NaN
2,QCL,Crops and livestock products,8,Albania,5111,Stocks,2111.0,Cattle,2010,2010,An,493000.0,A,Official figure,NaN
3,QCL,Crops and livestock products,8,Albania,5111,Stocks,2111.0,Cattle,2020,2020,An,362583.0,A,Official figure,NaN
4,QCL,Crops and livestock products,8,Albania,5510,Production,2165.0,Palm oil,2010,2010,t,0.0,M,"Missing value (data cannot exist, not applicable)",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2152,QCL,Crops and livestock products,716,Zimbabwe,5111,Stocks,2111.0,Cattle,2020,2020,An,5195154.0,A,Official figure,NaN
2153,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,1610.0,"Coffee, green",2010,2010,ha,1527.0,I,Imputed value,NaN
2156,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,1610.0,"Coffee, green",2020,2020,ha,2642.0,E,Estimated value,NaN
2159,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2010,2010,ha,42288.0,A,Official figure,NaN


In [81]:
def _get_commodity_expansion_rates(country, min_year=2010, max_year=2020): #TODO: Max year should be 2022, this is temporary
        
    FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
   'Natural rubber in primary forms']

    #TODO: this needs to be filtered so that for every commodity and year, there is only 1 element (either area, or production etc.)

    country_data = df_filtered.loc[df_filtered.Area == country]

    values = []

    for _class in FAOSTAT_classes:
        try:
            production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(min_year)), 'Value' ].item()
            production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(max_year)), 'Value' ].item()
            expansion = production_at_the_end - production_at_the_start
        except ValueError: #Signifies that one (or both) of the values are missing
            expansion = 0
        values.append(expansion)

    return values

In [86]:
vals = {}

for country in tropical_countries.name.unique():
    vals[country] = _get_commodity_expansion_rates(country)

In [88]:
FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
'Natural rubber in primary forms']

In [92]:
tmp_df = pd.DataFrame(vals).T

In [93]:
tmp_df.columns = FAOSTAT_classes

In [107]:
stds = tmp_df.std()

In [108]:
mean = tmp_df.mean()

In [109]:
mean

Cattle                             710911.610619
Palm oil                           266523.297168
Soya beans                         189918.168142
Cocoa beans                         17669.690265
Coffee, green                        7045.203540
Natural rubber in primary forms     29000.309735
dtype: float64

In [110]:
stds

Cattle                             3.040535e+06
Palm oil                           2.154836e+06
Soya beans                         1.360036e+06
Cocoa beans                        1.923138e+05
Coffee, green                      8.108818e+04
Natural rubber in primary forms    1.602785e+05
dtype: float64

In [97]:
tmp_df.loc[tmp_df.index == "Brazil"]

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms
Brazil,8295173.0,326763.0,13864342.0,-72210.0,-261356.0,38587.0


In [100]:
country_data = df_filtered.loc[df_filtered.Area == 'Chile']

In [101]:
country_data


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
338,QCL,Crops and livestock products,152,Chile,5111,Stocks,2111.0,Cattle,2010,2010,An,3830000.0,E,Estimated value,NaN
339,QCL,Crops and livestock products,152,Chile,5111,Stocks,2111.0,Cattle,2020,2020,An,3193276.0,I,Imputed value,NaN
340,QCL,Crops and livestock products,152,Chile,5312,Area harvested,141.0,Soya beans,2010,2010,ha,0.0,M,"Missing value (data cannot exist, not applicable)",NaN
342,QCL,Crops and livestock products,152,Chile,5312,Area harvested,141.0,Soya beans,2020,2020,ha,0.0,M,"Missing value (data cannot exist, not applicable)",NaN


In [103]:
df_filtered['Flag Description'].unique()

array(['Official figure', 'Estimated value',
       'Missing value (data cannot exist, not applicable)',
       'Imputed value', 'Figure from international organizations'],
      dtype=object)

In [115]:
tropical_countries.loc[tropical_countries.name == "Brazil"]

,pop_est,continent,name,iso_a3,gdp_md_est,geometry
29,211049527.0,South America,Brazil,BRA,1839758,"POLYGON ((-53.37366 -33.76838, -53.65054 -33.2..."
